# **ST 554 Spring 2026 Final Project**
## *Created by Cody Ashby on April 27, 2026*
### Fitting The Model

In [1]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from pyspark.sql.functions import window, col
from pyspark.sql.types import StructType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, Binarizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder # set parallelism to 128 when doing CV!
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 15:25:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 15:25:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 15:25:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/29 15:25:18 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
power_data = pd.read_csv("power_ml_data.csv")

In [3]:
spark_power_data = spark.read.csv("power_ml_data.csv",inferSchema=True,header=True)
spark_power_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

In [4]:
sqlTrans=SQLTransformer(statement="SELECT Month, Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, \
                                        Power_Zone_1, Power_Zone_2, Power_Zone_3 as label, CAST(Hour AS DOUBLE) AS Revised_Hour FROM __THIS__")

In [5]:
binary_Hour=Binarizer(threshold=6.5,inputCol="Revised_Hour",outputCol="Binary_Hour")

In [6]:
#Month_Index=StringIndexer(inputCol="Month",outputCol="Indexed_Month")
Month_OHE=OneHotEncoder(inputCol="Month",outputCol="Encoded_Month")

In [7]:
Weather_Vars=VectorAssembler(inputCols=['Temperature','Humidity','Wind_Speed','General_Diffuse_Flows','Diffuse_Flows'],outputCol='weather_features',handleInvalid='keep')
Weather_Comps=Weather_Vars.transform(spark_power_data)
PC_features=PCA(k=2,inputCol='weather_features',outputCol="Prin_Comps")
Fitted_PC=PC_features.fit(Weather_Comps)

In [8]:
total_features=VectorAssembler(inputCols=['Prin_Comps','Binary_Hour','Power_Zone_1','Power_Zone_2','Encoded_Month'],outputCol='features',handleInvalid='keep')

In [9]:
MLR=LinearRegression()
grand_pipeline=Pipeline(stages=[sqlTrans,binary_Hour,Month_OHE,Weather_Vars,PC_features,total_features,MLR])

In [10]:
EN_Param_Grid=ParamGridBuilder().addGrid(MLR.regParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]) \
                            .addGrid(MLR.elasticNetParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]).build()

In [13]:
ElasticNet_CrossVal=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=EN_Param_Grid,
                                   evaluator=RegressionEvaluator(metricName='rmse'),
                                   numFolds=5,
                                   seed=8,
                                   parallelism=128)
ElasticNet_Model=ElasticNet_CrossVal.fit(spark_power_data)

In [14]:
ElasticNet_RMSEs=[]
for _ in range(len(EN_Param_Grid)):
    ElasticNet_RMSEs.append([ElasticNet_Model.avgMetrics[_],EN_Param_Grid[_].values()])
ElasticNet_RMSEs

[[2147.743822079609, dict_values([0.0, 0.0])],
 [2147.743822079609, dict_values([0.0, 0.05])],
 [2147.743822079609, dict_values([0.0, 0.1])],
 [2147.743822079609, dict_values([0.0, 0.25])],
 [2147.743822079609, dict_values([0.0, 0.5])],
 [2147.743822079609, dict_values([0.0, 0.75])],
 [2147.743822079609, dict_values([0.0, 0.9])],
 [2147.743822079609, dict_values([0.0, 0.95])],
 [2147.743822079609, dict_values([0.0, 0.98])],
 [2147.743822079609, dict_values([0.0, 0.99])],
 [2147.743822079609, dict_values([0.0, 1.0])],
 [2147.743804871119, dict_values([0.05, 0.0])],
 [2147.745520060856, dict_values([0.05, 0.05])],
 [2147.738868265524, dict_values([0.05, 0.1])],
 [2147.7435245465463, dict_values([0.05, 0.25])],
 [2147.743723303225, dict_values([0.05, 0.5])],
 [2147.7424631014997, dict_values([0.05, 0.75])],
 [2147.7427017257346, dict_values([0.05, 0.9])],
 [2147.74267120382, dict_values([0.05, 0.95])],
 [2147.742633940907, dict_values([0.05, 0.98])],
 [2147.742635289277, dict_values([0.05

Upon close inspection of each RMSE associated with each pair of hyperparameters, we can see that the lowest RMSE goes to the one with the `regParam` of 0.05 and the `ElasticNetParam` of 0.1, indicating fairly minimal regularization required for this model. The CV error, as well as the training RMSE, for this model is shown below.

In [15]:
ElasticNet_CV_Error=np.mean(ElasticNet_Model.avgMetrics)
ElasticNet_Training_RMSE=min(ElasticNet_Model.avgMetrics)
print(ElasticNet_CV_Error)
print(ElasticNet_Training_RMSE)

2147.7574775784437
2147.738868265524


With those numbers in mind, we will now build our optimal Elastic Net model using the tuning parameters associated with the lowest RMSE.

In [16]:
Best_EN_Params=ParamGridBuilder().addGrid(MLR.regParam,[0.05]).addGrid(MLR.elasticNetParam,[0.1]).build()
Best_ElasticNet_CV=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=Best_EN_Params,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5)
Best_ElasticNet_Model=Best_ElasticNet_CV.fit(spark_power_data)

This fitted model will now be used to make predictions based on the original dataset. We'll also note how far each prediction is from the actual response with a `residuals` column, as shown below.

In [17]:
Best_ElasticNet_Model_Preds=Best_ElasticNet_Model.transform(spark_power_data)
Best_ElasticNet_Model_Preds_Resids=Best_ElasticNet_Model_Preds.withColumn("residuals",col("label")-col("prediction"))
Best_ElasticNet_Model_Preds_Resids.select("label","prediction","residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386| 20878.85066079531|-637.8868007953097|
|20131.08434|18660.227266546775| 1470.857073453226|
|19668.43373| 18204.75215311686|1463.6815768831402|
|18899.27711|17590.648498340925|1308.6286116590745|
|18442.40964|  16997.3019864581|1445.1076535419015|
|18130.12048|16517.686723495084|1612.4337565049173|
|17945.06024| 16093.24614105391| 1851.814098946088|
|17459.27711|15722.695360254002|1736.5817497459975|
|17025.54217|15271.043828662507| 1754.498341337494|
|16794.21687|14938.348764665025|1855.8681053349756|
|16638.07229|14652.383721423037| 1985.688568576963|
|16395.18072|14414.900662729378| 1980.280057270622|
|16117.59036|14082.889275685437|2034.7010843145636|
| 15822.6506|13624.882091434294|2197.7685085657067|
|15672.28916|13450.340775466415|2221.9483845335853|
|15597.10843| 13302.24639455996| 2294.862035440041|
|15510.36145

In [ ]:
#The overall idea of Best_ElasticNet_Model_Preds_Resids will be saved as a dataframe for later use. Namely, when we use incoming data to make predictions.

### Streaming Data

In [51]:
#power_streaming_data=pd.read_csv("Other_Files")

In [52]:
Read_Power_Stream=spark.readStream.schema(spark_power_data.schema).csv("Other_Files",header=True)

In [53]:
Streamed_Preds=Best_ElasticNet_Model.transform(Read_Power_Stream)
Streamed_Preds_Renamed=Read_Power_Stream.withColumnRenamed("Power_Zone_3","label") \
                                        .select("label","Temperature","Humidity","Wind_Speed","General_Diffuse_Flows", \
                                                "Diffuse_Flows","Power_Zone_1","Power_Zone_2","Month","Hour")
Streamed_Preds_Resids=Streamed_Preds.withColumn("residual",col("label")-col("prediction")).select("label","prediction","residual")
Joined_Streams=Streamed_Preds_Resids.join(Streamed_Preds_Renamed,"label","inner")

In [54]:
Power_Query=Joined_Streams.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17447.71084| 17667.24386214502|-219.53302214502037|       8.49|    76.9|     0.083|                0.084|        0.141| 28362.53165| 18320.97264|    1|   0|
|16029.58794|16727.599468623455| -698.0115286234559|      16.32|   52.15|     0.083|                603.7|        160.5| 32570.84746| 20677.20365|    2|  11|
|15381.29032|16410.605191945462| -1029.314871945462|      15.49|   63.14|     0.083|                324.0|       

### Producing Data

Now that the query has started, let's import the file that allows us to view the output.

In [57]:
%run Other_Files/PowerStreamDataProducer.py

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|    24750.0|       NaN|     NaN|     4767.0|   17.98|     62.75|                 4.92|        257.3|       49.77| 31046.15385| NULL|  11|
|31872.35348|       NaN|     NaN|     4906.0|   11.26|      72.8|                0.081|        0.055|        0.13| 36325.47529| NULL|  12|
|26656.34675|       NaN|     NaN|     1988.0|   20.06|      81.9|                0.068|         25.6|       19.88| 44241.83607| NULL|   5|
| 19591.0387|       NaN|     NaN|     1488.0|   20.29|      51.4|    

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24969.42149|       NaN|     NaN|     4391.0|   23.27|     55.04|                0.072|        353.8|       345.4| 30295.38462| NULL|  11|
|13342.24924|       NaN|     NaN|      610.0|   12.15|      76.1|                4.916|        0.029|       0.167|  22564.0678| NULL|   2|
|30288.60759|       NaN|     NaN|     2988.0|   34.27|     31.01|                4.909|        833.0|        93.6|  40779.2691| NULL|   7|
|19000.41322|       NaN|     NaN|     4764.0|   16.26|     67.72|    

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14779.33131|       NaN|     NaN|      769.0|   13.43|      71.1|                0.075|        0.029|       0.119|  23637.9661| NULL|   2|
|17234.04255|       NaN|     NaN|      651.0|    8.47|      86.3|                 0.08|         76.2|        75.2| 27341.69492| NULL|   2|
|31161.70604|       NaN|     NaN|     5030.0|   14.64|      78.8|                0.086|        0.048|       0.145| 38211.40684| NULL|  12|
|21559.87842|       NaN|     NaN|       24.0|   15.59|     55.76|    

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|16547.36842|       NaN|     NaN|     1931.0|    17.9|      84.2|                4.916|        0.062|       0.104| 26640.78689| NULL|   5|
|31181.01266|       NaN|     NaN|     2819.0|   30.13|     37.48|                4.922|        0.106|       0.089| 48905.78073| NULL|   7|
|16884.29752|       NaN|     NaN|     4523.0|   18.27|     68.01|                 0.07|        0.062|       0.082| 22516.92308| NULL|  11|
|25321.87788|       NaN|     NaN|     5094.0|   16.37|      74.8|    

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
| 19901.0395|       NaN|     NaN|     2217.0|   23.16|     67.86|                 0.07|        866.0|       127.7| 36619.86755| NULL|   6|
|23597.92531|       NaN|     NaN|     4192.0|    20.1|      79.6|                4.918|        0.051|         0.1| 37742.49453| NULL|  10|
|20822.72727|       NaN|     NaN|     4630.0|   15.17|     55.07|                0.084|         0.04|       0.082| 25907.69231| NULL|  11|
|    22800.0|       NaN|     NaN|      287.0|    13.6|     48.35|    

...

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|28199.36642|       NaN|     NaN|     3299.0|    26.6|     66.36|                0.068|        792.0|        62.9| 40422.28635| NULL|   8|
|22343.15353|       NaN|     NaN|     4250.0|   20.06|      78.4|                4.922|         0.08|       0.082| 34875.09847| NULL|  10|
|33437.25069|       NaN|     NaN|     4971.0|   12.28|     65.69|                0.078|        4.143|       4.221| 39458.55513| NULL|  12|
|20726.14108|       NaN|     NaN|     3933.0|   26.56|     46.48|   

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|18557.23014|       NaN|     NaN|     1656.0|   20.48|     49.05|                0.085|        459.1|       195.0| 32352.72336| NULL|   4|
|27330.09119|       NaN|     NaN|      330.0|   12.02|      72.1|                0.087|        0.048|       0.111| 44536.70886| NULL|   1|
| 28343.8226|       NaN|     NaN|     3129.0|   28.62|     63.79|                4.903|        231.5|       187.3| 40473.42952| NULL|   8|
|30288.60759|       NaN|     NaN|     2988.0|   34.27|     31.01|   

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25731.39293|       NaN|     NaN|     2436.0|   20.98|      81.9|                0.064|        0.077|       0.111| 42774.03974| NULL|   6|
|21266.94387|       NaN|     NaN|     3810.0|   24.81|     60.19|                4.933|        356.8|        92.9|  35929.9115| NULL|   9|
|18834.51143|       NaN|     NaN|     3584.0|   23.48|      76.7|                 0.24|        275.6|        99.2| 28404.95575| NULL|   9|
|22534.17722|       NaN|     NaN|     2705.0|   28.95|     62.53|   

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13251.21951|       NaN|     NaN|      968.0|   13.01|      72.6|                0.081|        0.037|       0.141| 23095.14894| NULL|   3|
|24787.19008|       NaN|     NaN|     4703.0|   18.68|     53.32|                0.081|        429.0|       37.78| 30326.15385| NULL|  11|
|16133.47413|       NaN|     NaN|     3203.0|    25.0|     56.52|                4.902|         0.08|       0.081| 23583.39623| NULL|   8|
|23226.75667|       NaN|     NaN|     5044.0|    17.3|     61.45|   

-------------------------------------------
Batch: 20
-------------------------------------------
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|prediction|residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+----------+--------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25924.39024|       NaN|     NaN|     1016.0|   16.47|     61.62|                0.084|        0.062|       0.096| 43285.78723| NULL|   3|
|20255.10836|       NaN|     NaN|     1788.0|   25.23|     24.49|                4.921|        320.1|       347.6| 32293.77049| NULL|   5|
|29582.27848|       NaN|     NaN|     2650.0|   24.75|      80.3|                4.918|        0.563|       0.463| 46717.87375| NULL|   7|
|15982.95218|       NaN|     NaN|     3678.0|   21.01|     66.69|   

In [ ]:
#Power_Query.stop() will be used to stop the query (if need be).